# **Explore StatsBomb data — 2015/16 Big 5 leagues**

First look at the StatsBomb open data for the 2015/16 season (Premier League, La Liga, Bundesliga, Serie A, Ligue 1), to see what columns are available in the events dataframe before building the aggression score.

In [1]:
from statsbombpy import sb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

# Plot style: real LaTeX when available (best-looking, e.g. local machines with a LaTeX
# install), a LaTeX-free fallback otherwise (Binder/Colab, which don't have LaTeX)
import shutil
import scienceplots

if shutil.which('latex'):
    plt.style.use(['science', 'nature'])
else:
    plt.style.use(['science', 'no-latex', 'nature'])
    plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'Liberation Sans', 'DejaVu Sans']
plt.rcParams['figure.figsize'] = [5, (4.8/6.4)*5]
plt.rcParams['figure.dpi'] = 150

# Nature style pins these to 7pt by default; bump for legibility
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 9
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 9
plt.rcParams['legend.fontsize'] = 9


## Big 5 leagues, 2015/16 season

`season_id` 27 is 2015/2016 across competitions. The Big 5 `competition_id`s are: 9 (Bundesliga), 11 (La Liga), 7 (Ligue 1), 2 (Premier League), 12 (Serie A).

In [2]:
BIG5_2015_16 = {
    "Bundesliga": (9, 27),
    "La Liga": (11, 27),
    "Ligue 1": (7, 27),
    "Premier League": (2, 27),
    "Serie A": (12, 27),
}

competitions = sb.competitions()
competitions[competitions["season_id"] == 27]

/Users/slimane/Documents/GitHub/football-aggression-index/.venv-football-aggression-index/lib/python3.13/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
1,9,27,Germany,1. Bundesliga,male,False,False,2015/2016,2024-05-19T11:11:14.192381,NaN,NaN,2024-05-19T11:11:14.192381
6,16,27,Europe,Champions League,male,False,False,2015/2016,2024-06-12T07:45:38.786894,2021-06-13T16:17:31.694,NaN,2024-06-12T07:45:38.786894
45,11,27,Spain,La Liga,male,False,False,2015/2016,2025-04-23T13:59:22.835792,2021-06-13T16:17:31.694,NaN,2025-04-23T13:59:22.835792
63,7,27,France,Ligue 1,male,False,False,2015/2016,2026-05-12T22:46:54.722768,NaN,NaN,2026-05-12T22:46:54.722768
68,2,27,England,Premier League,male,False,False,2015/2016,2025-12-17T14:38:00.447595,2021-06-13T16:17:31.694,NaN,2025-12-17T14:38:00.447595
70,12,27,Italy,Serie A,male,False,False,2015/2016,2025-08-15T14:28:50.169562,NaN,NaN,2025-08-15T14:28:50.169562


In [3]:
matches_per_league = pd.Series(
    {
        league: len(sb.matches(competition_id=comp_id, season_id=season_id))
        for league, (comp_id, season_id) in BIG5_2015_16.items()
    },
    name="num_matches",
).sort_index()

total_matches = matches_per_league.sum()

matches_per_league

/Users/slimane/Documents/GitHub/football-aggression-index/.venv-football-aggression-index/lib/python3.13/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


/Users/slimane/Documents/GitHub/football-aggression-index/.venv-football-aggression-index/lib/python3.13/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


/Users/slimane/Documents/GitHub/football-aggression-index/.venv-football-aggression-index/lib/python3.13/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


Bundesliga         34
La Liga           380
Ligue 1           377
Premier League    380
Serie A           380
Name: num_matches, dtype: int64

In [4]:
total_matches

np.int64(1551)

## Pull one match's events to inspect the event dataframe

Using the Premier League as a starting point.

In [5]:
competition_id, season_id = BIG5_2015_16["Premier League"]
matches = sb.matches(competition_id=competition_id, season_id=season_id)
matches.head()

/Users/slimane/Documents/GitHub/football-aggression-index/.venv-football-aggression-index/lib/python3.13/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


,match_id,match_date,kick_off,home_score,away_score,match_status,match_status_360,last_updated,last_updated_360,match_week,competition_id,competition_country_name,competition_name,competition,season_id,season,home_team_id,home_team,home_team_gender,home_team_group,home_team_country_id,home_team_country_name,away_team_id,away_team,away_team_gender,away_team_group,away_team_country_id,away_team_country_name,competition_stage_id,competition_stage,stadium_id,stadium,stadium_country_id,stadium_country_name,referee_id,referee,referee_country_id,referee_country_name,home_managers,away_managers,home_manager_id,home_manager_name,home_manager_nickname,home_manager_dob,home_manager_country_id,home_manager_country_name,away_manager_id,away_manager_name,away_manager_nickname,away_manager_dob,away_manager_country_id,away_manager_country_name,data_version,shot_fidelity_version,xy_fidelity_version
0,3754217,2015-09-19,11:45:00.000,2,0,available,scheduled,2025-12-16T17:01:18.696515,2021-06-13T16:17:31.694,6,2,England,Premier League,England - Premier League,27,2015/2016,33,Chelsea,male,None,68,England,1,Arsenal,male,None,68,England,1,Regular Season,10,Stamford Bridge,68,England,6,Mike Dean,68,England,José Mario Felix dos Santos Mourinho,Arsène Wenger,383,José Mario Felix dos Santos Mourinho,José Mourinho,1963-01-26,183,Portugal,577,Arsène Wenger,NaN,1949-10-22,78,France,1.1.0,2,2
1,3754117,2015-12-13,12:30:00.000,0,2,available,scheduled,2025-12-16T17:04:24.176407,2021-06-13T16:17:31.694,16,2,England,Premier League,England - Premier League,27,2015/2016,59,Aston Villa,male,None,68,England,1,Arsenal,male,None,68,England,1,Regular Season,211,Villa Park,68,England,12,Kevin Friend,68,England,Rémi Garde,Arsène Wenger,92,Rémi Garde,NaN,1966-04-03,78,France,577,Arsène Wenger,NaN,1949-10-22,78,France,1.1.0,2,2
2,3754296,2015-12-21,19:00:00.000,2,1,available,scheduled,2025-12-17T11:59:35.059270,2021-06-13T16:17:31.694,17,2,England,Premier League,England - Premier League,27,2015/2016,1,Arsenal,male,None,68,England,36,Manchester City,male,None,68,England,1,Regular Season,3,Emirates Stadium,68,England,5,Andre Marriner,68,England,Arsène Wenger,Manuel Luis Pellegrini Ripamonti,577,Arsène Wenger,NaN,1949-10-22,78,France,733,Manuel Luis Pellegrini Ripamonti,Manuel Pellegrini,1953-09-16,45,Chile,1.1.0,2,2
3,3753983,2015-10-31,14:00:00.000,0,3,available,scheduled,2025-12-16T17:02:24.849544,2021-06-13T16:17:31.694,11,2,England,Premier League,England - Premier League,27,2015/2016,26,Swansea City,male,None,249,Wales,1,Arsenal,male,None,68,England,1,Regular Season,45,Swansea.com Stadium,249,Wales,12,Kevin Friend,68,England,Garry Monk,Arsène Wenger,270,Garry Monk,NaN,1979-03-06,68,England,577,Arsène Wenger,NaN,1949-10-22,78,France,1.1.0,2,2
4,3754160,2015-12-05,14:00:00.000,3,1,available,scheduled,2025-12-16T17:03:25.661537,2021-06-13T16:17:31.694,15,2,England,Premier League,England - Premier League,27,2015/2016,1,Arsenal,male,None,68,England,41,Sunderland,male,None,68,England,1,Regular Season,3,Emirates Stadium,68,England,9,Robert Madley,68,England,Arsène Wenger,Sam Allardyce,577,Arsène Wenger,NaN,1949-10-22,78,France,561,Sam Allardyce,NaN,1954-10-19,68,England,1.1.0,2,2


In [6]:
match_id = matches.iloc[0]["match_id"]
events = sb.events(match_id=match_id)
events.head()

/Users/slimane/Documents/GitHub/football-aggression-index/.venv-football-aggression-index/lib/python3.13/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


,bad_behaviour_card,ball_receipt_outcome,ball_recovery_recovery_failure,block_deflection,carry_end_location,clearance_aerial_won,clearance_body_part,clearance_head,clearance_left_foot,clearance_other,clearance_right_foot,counterpress,dribble_outcome,dribble_overrun,duel_outcome,duel_type,duration,foul_committed_advantage,foul_committed_card,foul_committed_type,foul_won_advantage,foul_won_defensive,goalkeeper_body_part,goalkeeper_end_location,goalkeeper_outcome,goalkeeper_position,goalkeeper_technique,goalkeeper_type,id,index,injury_stoppage_in_chain,interception_outcome,location,match_id,minute,miscontrol_aerial_won,off_camera,out,pass_aerial_won,pass_angle,pass_assisted_shot_id,pass_body_part,pass_cross,pass_cut_back,pass_deflected,pass_end_location,pass_goal_assist,pass_height,pass_inswinging,pass_length,pass_outcome,pass_outswinging,pass_recipient,pass_recipient_id,pass_shot_assist,pass_switch,pass_technique,pass_through_ball,pass_type,period,play_pattern,player,player_id,position,possession,possession_team,possession_team_id,related_events,second,shot_aerial_won,shot_body_part,shot_end_location,shot_first_time,shot_freeze_frame,shot_key_pass_id,shot_one_on_one,shot_outcome,shot_statsbomb_xg,shot_technique,shot_type,substitution_outcome,substitution_outcome_id,substitution_replacement,substitution_replacement_id,tactics,team,team_id,timestamp,type,under_pressure
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9d86a178-3514-45d1-9d14-1372e846d17b,1,NaN,NaN,NaN,3754217,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,Regular Play,NaN,NaN,NaN,1,Chelsea,33,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'formation': 4231, 'lineup': [{'player': {'id...",Chelsea,33,00:00:00.000,Starting XI,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,82acc213-90f3-4fee-8305-bd9e403cec42,2,NaN,NaN,NaN,3754217,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,Regular Play,NaN,NaN,NaN,1,Chelsea,33,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'formation': 4231, 'lineup': [{'player': {'id...",Arsenal,1,00:00:00.000,Starting XI,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,d18f1a33-65ba-42e5-a9fd-bbfc709694e8,3,NaN,NaN,NaN,3754217,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,Regular Play,NaN,NaN,NaN,1,Chelsea,33,[49147091-455c-4372-8d56-96e1ca71d01a],0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Chelsea,33,00:00:00.000,Half Start,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49147091-455c-4372-8d56-96e1ca71d01a,4,NaN,NaN,NaN,3754217,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,Regular Play,NaN,NaN,NaN,1,Chelsea,33,[d18f1a33-65ba-42e5-a9fd-bbfc709694e8],0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Arsenal,1,00:00:00.000,Half Start,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7457b6d0-10f8-48ea-9b80-8c0703914ece,1718,NaN,NaN,NaN,3754217,45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,From Throw In,NaN,NaN,NaN,92,Arsenal,1,[d9259d49-479e-4faa-aa7c-5b8fc586c8b2],0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Arsenal,1,00:00:00.000,Half Start,NaN


## All available columns

Full column listing, to know what's available for building PPDA, pressures, counterpressing, defensive-line, and fouls components.

In [7]:
len(events)

3732

In [8]:
events.columns.tolist()

['bad_behaviour_card',
 'ball_receipt_outcome',
 'ball_recovery_recovery_failure',
 'block_deflection',
 'carry_end_location',
 'clearance_aerial_won',
 'clearance_body_part',
 'clearance_head',
 'clearance_left_foot',
 'clearance_other',
 'clearance_right_foot',
 'counterpress',
 'dribble_outcome',
 'dribble_overrun',
 'duel_outcome',
 'duel_type',
 'duration',
 'foul_committed_advantage',
 'foul_committed_card',
 'foul_committed_type',
 'foul_won_advantage',
 'foul_won_defensive',
 'goalkeeper_body_part',
 'goalkeeper_end_location',
 'goalkeeper_outcome',
 'goalkeeper_position',
 'goalkeeper_technique',
 'goalkeeper_type',
 'id',
 'index',
 'injury_stoppage_in_chain',
 'interception_outcome',
 'location',
 'match_id',
 'minute',
 'miscontrol_aerial_won',
 'off_camera',
 'out',
 'pass_aerial_won',
 'pass_angle',
 'pass_assisted_shot_id',
 'pass_body_part',
 'pass_cross',
 'pass_cut_back',
 'pass_deflected',
 'pass_end_location',
 'pass_goal_assist',
 'pass_height',
 'pass_inswinging',

In [9]:
len(events.columns.tolist())

90

In [10]:
events["type"].value_counts()

type
Pass                 1046
Ball Receipt*         963
Carry                 814
Pressure              320
Ball Recovery         106
Duel                   78
Dribble                52
Clearance              42
Block                  41
Goal Keeper            34
Dribbled Past          33
Shot                   33
Foul Committed         32
Foul Won               29
Miscontrol             27
Dispossessed           26
Interception           20
Substitution            6
Half Start              4
Injury Stoppage         4
Half End                4
Shield                  3
Bad Behaviour           3
Tactical Shift          3
Starting XI             2
Referee Ball-Drop       2
Player Off              1
Player On               1
Error                   1
Own Goal For            1
Own Goal Against        1
Name: count, dtype: int64

## Filtering the relevant columns for our aggression score

In [11]:
score_cols = [
    # join / context
    "match_id", "period", "minute", "second",
    "type", "team", "possession", "possession_team", "location",
    # PPDA
    "pass_outcome",
    "duel_type", "duel_outcome",
    "interception_outcome",
    # counterpressing
    "counterpress",
    # fouls
    "foul_committed_type",
]

events_slim = events[score_cols].copy()
events_slim.head(20)


,match_id,period,minute,second,type,team,possession,possession_team,location,pass_outcome,duel_type,duel_outcome,interception_outcome,counterpress,foul_committed_type
0,3754217,1,0,0,Starting XI,Chelsea,1,Chelsea,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3754217,1,0,0,Starting XI,Arsenal,1,Chelsea,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3754217,1,0,0,Half Start,Chelsea,1,Chelsea,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3754217,1,0,0,Half Start,Arsenal,1,Chelsea,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3754217,2,45,0,Half Start,Arsenal,92,Arsenal,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,3754217,2,45,0,Half Start,Chelsea,92,Arsenal,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,3754217,1,0,0,Pass,Arsenal,2,Arsenal,"[61.0, 40.1]",NaN,NaN,NaN,NaN,NaN,NaN
7,3754217,1,0,1,Pass,Arsenal,2,Arsenal,"[59.3, 42.1]",NaN,NaN,NaN,NaN,NaN,NaN
8,3754217,1,0,5,Pass,Arsenal,2,Arsenal,"[49.5, 41.7]",NaN,NaN,NaN,NaN,NaN,NaN
9,3754217,1,0,7,Pass,Arsenal,2,Arsenal,"[65.9, 48.7]",Incomplete,NaN,NaN,NaN,NaN,NaN
